In [1]:
# This code classifies bird sounds in audio files with birds in them.

import librosa
import numpy as np
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [2]:
# Lists the types of birds

birds = []

taxonomy_df = pd.read_csv("taxonomy.csv")

for row in taxonomy_df.itertuples():
    birds += (row.class_name == 'Aves')*[row.primary_label]

In [3]:
len(birds) # There are 162 different bird species.

162

In [4]:
print(birds)

['ashgre1', 'astcra1', 'bafcur1', 'baffal1', 'banana', 'barant1', 'batbel1', 'baymac', 'bbwduc', 'bcwfin2', 'bkcdon', 'bkhpar', 'blchaw1', 'blheag1', 'blttit1', 'bncfly', 'bobfly1', 'brcmar1', 'brnowl', 'bucmot4', 'bucpar', 'bufpar', 'bunibi1', 'burowl', 'camfli1', 'chacha1', 'chbmoc1', 'chobla1', 'chvcon1', 'cibspi1', 'coffal1', 'compau', 'compot1', 'crbthr1', 'crebec1', 'dwatin1', 'epaori4', 'eulfly1', 'fabwre1', 'fepowl', 'ficman1', 'flawar1', 'fotfly', 'fusfly1', 'gilhum1', 'giwrai1', 'glteme1', 'grasal3', 'greani1', 'greant1', 'greela', 'grekis', 'grepot1', 'gretho2', 'greyel', 'grfdov1', 'grhtan1', 'gycwor1', 'horscr1', 'houspa', 'hyamac1', 'larela1', 'lesela1', 'lesgrf1', 'limpki', 'linwoo1', 'litcuc2', 'litnig1', 'mabpar', 'magant1', 'magtan2', 'masgna1', 'nacnig1', 'ocecra1', 'oliwoo1', 'orbtro3', 'orwpar', 'osprey', 'pabspi1', 'palhor3', 'paltan1', 'phecuc1', 'picpig2', 'pirfly1', 'plasla1', 'platyr1', 'plcjay1', 'pluibi1', 'purjay1', 'pvttyr1', 'ragmac1', 'rebscy1', 'recfin1

In [5]:
train_soundscapes_labels_df = pd.read_csv("train_soundscapes_labels.csv")

train_soundscapes_reduced_df = pd.DataFrame({
    "filename": train_soundscapes_labels_df.iloc[:, :2].astype(str).agg(''.join, axis=1),
    "primary_label": train_soundscapes_labels_df.iloc[:, 3]
})

In [6]:
# Lists all files in train_soundscapes_reduced_df which contain birds.
# Organizes all birds in these files.

train_soundscapes_birds_df = pd.DataFrame(columns = ['filename', 'primary_label'])

for entry in train_soundscapes_reduced_df.itertuples():
    found_bird = False
    primary_labels = entry.primary_label.split(';')
    row_birds = ''
    
    for label in primary_labels:
        if label in birds:
            if found_bird:
                row_birds += ';'
            found_bird = True
            row_birds += label
    
    if found_bird:
        new_row = pd.DataFrame({'filename': [entry.filename], 'primary_label': [row_birds]})
        train_soundscapes_birds_df = pd.concat([train_soundscapes_birds_df, new_row], ignore_index=True)

In [7]:
train_soundscapes_birds_df.head()

,filename,primary_label
0,BC2026_Train_0019_S22_20211104_200000.ogg00:00:25,bunibi1
1,BC2026_Train_0019_S22_20211104_200000.ogg00:00:30,bunibi1
2,BC2026_Train_0019_S22_20211104_200000.ogg00:00:25,bunibi1
3,BC2026_Train_0019_S22_20211104_200000.ogg00:00:30,bunibi1
4,BC2026_Train_0027_S22_20211129_014500.ogg00:00:00,trsowl


In [8]:
# It is very common to have a sound file with multiple bird species.
# 228 out of 424 files in train_soundscapes_birds_df have multiple birds.

multiple_count = 0

for entry in train_soundscapes_birds_df.itertuples():
    if ';' in entry.primary_label:
        multiple_count += 1

print(multiple_count)

228


In [9]:
len(train_soundscapes_birds_df)

424

In [10]:
# In animal_class_classifier.ipynb, we created train_audio_df.csv.
# Each row in this csv corresponds to an audio file in the train_audio folder along with its class.
# Recall that each file in the train_audio folder has exactly one animal in it.
# This loop identifies all bird files in train_audio_df and writes their primary labels.
# Note that the primary label is just the part of the filename that occurs before the '/'.

train_audio_df = pd.read_csv('train_audio_df.csv')

bird_audio_df = pd.DataFrame(columns = ['filename', 'primary_label'])

for entry in train_audio_df.itertuples():
    if entry.primary_label == 'Aves':
        new_row = pd.DataFrame({'filename': [entry.filename], 'primary_label': [entry.filename.split('/')[0]]})
        bird_audio_df = pd.concat([bird_audio_df, new_row], ignore_index=True)

In [11]:
len(bird_audio_df) # 32254 files!

32254

In [12]:
bird_audio_df.head()

,filename,primary_label
0,crebec1/XC119358.ogg,crebec1
1,crebec1/XC602873.ogg,crebec1
2,crebec1/XC844032.ogg,crebec1
3,crebec1/XC611358.ogg,crebec1
4,crebec1/XC995496.ogg,crebec1


In [13]:
train_soundscapes_birds_df.to_csv('train_soundscapes_birds_df.csv', index=False)
bird_audio_df.to_csv('bird_audio_df.csv', index=False)

In [14]:
# We count the number of files in each class.

counter = {}
for bird in birds:
    counter[bird] = 0

for label in train_soundscapes_birds_df.primary_label:
    labels = label.split(';')
    for primary_label in labels:
        counter[primary_label] += 1

for label in bird_audio_df.primary_label:
    counter[str(label)] += 1

In [15]:
common_birds = []
rare_birds = []

threshold = 100

for bird in birds:
    if counter[bird] < threshold:
        rare_birds += [bird]
    else:
        common_birds += [bird]

In [16]:
len(common_birds), len(rare_birds) # There are 112 common birds and 50 rare birds.

(112, 50)

In [17]:
# We reduce train_soundscapes_birds and bird_audio_df into dataframes that only include the common birds.

train_soundscapes_common_birds_df = pd.DataFrame(columns = ['filename', 'primary_label'])

for entry in train_soundscapes_birds_df.itertuples():
    found_common_bird = False
    primary_labels = entry.primary_label.split(';')
    row_common_birds = ''
    
    for label in primary_labels:
        if label in common_birds:
            if found_common_bird:
                row_common_birds += ';'
            found_common_bird = True
            row_common_birds += label
    
    if found_common_bird:
        new_row = pd.DataFrame({'filename': [entry.filename], 'primary_label': [row_common_birds]})
        train_soundscapes_common_birds_df = pd.concat([train_soundscapes_common_birds_df, new_row], ignore_index=True)

In [ ]:
# All but 18 of the files in train_soundscapes_birds had a common bird.

print(len(train_soundscapes_birds_df), len(train_soundscapes_common_birds_df))

424 406


In [21]:
train_soundscapes_common_birds_df.head()

,filename,primary_label
0,BC2026_Train_0019_S22_20211104_200000.ogg00:00:25,bunibi1
1,BC2026_Train_0019_S22_20211104_200000.ogg00:00:30,bunibi1
2,BC2026_Train_0019_S22_20211104_200000.ogg00:00:25,bunibi1
3,BC2026_Train_0019_S22_20211104_200000.ogg00:00:30,bunibi1
4,BC2026_Train_0027_S22_20211129_014500.ogg00:00:00,trsowl


In [22]:
common_bird_audio_df = pd.DataFrame(columns = ['filename', 'primary_label'])

for entry in bird_audio_df.itertuples():
    if str(entry.primary_label) in common_birds:
        new_row = pd.DataFrame({'filename': [entry.filename], 'primary_label': [str(entry.primary_label)]})
        common_bird_audio_df = pd.concat([common_bird_audio_df, new_row], ignore_index=True)

In [23]:
print(len(bird_audio_df), len(common_bird_audio_df)) # 28893 out 32254 bird files have a common bird (89.6%)

32254 28893


In [24]:
common_bird_audio_df.head()

,filename,primary_label
0,eulfly1/XC674270.ogg,eulfly1
1,eulfly1/iNat1663000.ogg,eulfly1
2,eulfly1/XC838488.ogg,eulfly1
3,eulfly1/XC1068975.ogg,eulfly1
4,eulfly1/XC456280.ogg,eulfly1


In [26]:
# We now have all the audio files (train_soundscapes_common_birds_df and common_bird_audio_df).

NUM_CLASSES = len(common_birds)
SR = 32000
N_MELS = 128
X_multi = []
y_multi = []

# This loop converts each audio file in train_soundscapes_common_birds_df to a mel spectrogram.

for row in train_soundscapes_common_birds_df.itertuples():
    filename = row.filename
    labels = row.primary_label.split(';')
    file = filename[:-8]
    start = int(filename[-2:])
    
    audio, _ = librosa.load(
        f"train_soundscapes/{file}",
        sr=SR,
        offset=start,
        duration=5
    )

# Converts the audio to a mel spectrogram, and normalizes it to the range [0, 1].

    mel = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    X_multi.append(mel_db[..., np.newaxis])
    
    # 🔥 MULTI-LABEL TARGET
    label_vector = [0] * NUM_CLASSES
    for i, bird in enumerate(common_birds):
        if bird in labels:
            label_vector[i] = 1

    y_multi.append(label_vector)
    
X_multi = np.array(X_multi, dtype=np.float32)
y_multi = np.array(y_multi, dtype=np.float32)

# X is the list of spectrograms, and y is the list of multi-label vectors.

In [27]:
len(y_multi)

406

In [37]:
# We repeat the same process for the audio files in common_bird_audio_df.
# Because we have so many files, we select random 5 second chunks from each of them.
# This code takes 12 minutes to run.

X_single = []
y_single = []

EXPECTED_FRAMES = int(np.ceil((SR * 5) / 512))

for row in common_bird_audio_df.itertuples():
    filename = row.filename
    label = row.primary_label
    audio, _ = librosa.load(
        f"train_audio/{filename}",
        sr=SR
    )

# After breaking the audio files into 5 second chunks, there were far too many files.
# Rather than processing every file, we take a few chunks.
# If the file is less than 15 seconds, we take 1 chunk. Otherwise, we take 2.

    chunk_length = 5 * SR

    if len(audio) < 2 * chunk_length:
        # 5–10 sec → just take first chunk
        audio_clips = [audio[:chunk_length]]

    elif len(audio) < 3 * chunk_length:
        # 10–15 sec → allow overlap (random is fine)
        audio_clips = []
        for _ in range(2):
            start = np.random.randint(0, len(audio) - chunk_length)
            audio_clips.append(audio[start:start + chunk_length])

    else:
        # ≥15 sec → enforce separation safely
        start1 = np.random.randint(0, len(audio) - chunk_length)
        start2 = np.random.randint(0, len(audio) - chunk_length)

        # Try a few times only (avoid slow loops)
        for _ in range(5):
            if abs(start1 - start2) >= chunk_length:
                break
            start2 = np.random.randint(0, len(audio) - chunk_length)

        audio_clips = [audio[start1:start1 + chunk_length], audio[start2:start2 + chunk_length]]

# Converts the audio files in audio_clips to a mel spectrogram, and normalizes it to the range [0, 1].

    for audio_clip in audio_clips:

        mel = librosa.feature.melspectrogram(
            y=audio_clip,
            sr=SR,
            n_mels=N_MELS
        )

        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min())

        # Pad / trim
        if mel_db.shape[1] < EXPECTED_FRAMES:
            mel_db = np.pad(mel_db, ((0, 0), (0, EXPECTED_FRAMES - mel_db.shape[1])))
        else:
            mel_db = mel_db[:, :EXPECTED_FRAMES]

        X_single.append(mel_db[..., np.newaxis])

        label_vector = [0] * NUM_CLASSES
        for j, bird in enumerate(common_birds):
            if bird == label:
                label_vector[j] = 1

        y_single.append(label_vector)

X_single = np.array(X_single, dtype=np.float32)
y_single = np.array(y_single, dtype=np.float32)

In [38]:
label

'fepowl'

In [39]:
print(len(y_single)) # We have 424 multi-class files and 53006 single class files.

53006


In [40]:
X = np.concatenate([X_multi, X_single], axis=0)
y = np.concatenate([y_multi, y_single], axis=0)

In [41]:
len(X)

53412

In [42]:
print((y.sum(axis=0) == 0).sum())

0


In [ ]:
print((y_single.sum(axis=0) == 0).sum())

0


In [44]:
# We shuffle the elements of X and y so that we do not over-sample certain types of data.

indices = np.arange(len(X))
np.random.shuffle(indices)

X = X[indices]
y = y[indices]

np.save("X_bird.npy", X)
np.save("y_bird.npy", y)

In [2]:
# To load the arrays, use this block.

X = np.load("X_bird.npy")
y = np.load("y_bird.npy")

In [ ]:
# This block breaks X into smaller pieces so that we can run the rest of the code in Google Colab.
# My laptop did not have the power to fit the model below.

# bird_chunks is a folder. Each file in the folder contains 1000 elements of X.

# This is about 16 gigabytes of data in total.
# Currently, I cannot save all this in Google Drive.

import os

os.makedirs("bird_chunks", exist_ok=True)

chunk_size = 1000

for i in range(0, len(X), chunk_size):
    np.save(f"bird_chunks/X_{i}.npy", X[i:i+chunk_size])
    np.save(f"bird_chunks/y_{i}.npy", y[i:i+chunk_size])

In [3]:
# SpecAugment introduces some light noise into our training data to make it more robust.

def spec_augment(mel, p=0.3):
    if np.random.rand() > p:
        return mel

    mel = mel.copy()
    n_mels, n_frames = mel.shape

    # light time mask
    t = np.random.randint(5, min(20, n_frames // 5))
    t0 = np.random.randint(0, n_frames - t)
    mel[:, t0:t0+t] = mel.mean()

    # light freq mask
    f = np.random.randint(3, min(10, n_mels // 5))
    f0 = np.random.randint(0, n_mels - f)
    mel[f0:f0+f, :] = mel.mean()

    return mel

In [4]:
def preprocess(mel):
    if np.random.rand() < 0.3:
        mel = spec_augment(mel)
    return mel

In [6]:
import sys
print(sys.executable)

/Users/noah/tf_env/bin/python


In [7]:
!pip install iterative-stratification

  Using cached iterative_stratification-0.1.9-py3-none-any.whl.metadata (1.3 kB)
Using cached iterative_stratification-0.1.9-py3-none-any.whl (8.5 kB)

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [5]:
# Performs a train-test-validation split.
# Ensures that every class has enough data.

from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)

train_idx, val_idx = next(mskf.split(X, y))

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

In [6]:
len(X)

53412

In [7]:
print((y_train.sum(axis=0) == 0).sum())

0


In [8]:
print((y_val.sum(axis=0) == 0).sum())

0


In [9]:
print((y.sum(axis=0) == 0).sum())

0


In [10]:
class_counts = y.sum(axis=0)

print("Total classes:", len(class_counts))
print("Nonzero classes:", (class_counts > 0).sum())
print("Zero classes:", (class_counts == 0).sum())

Total classes: 112
Nonzero classes: 112
Zero classes: 0


In [14]:
# We use a fairly large CNN because we have a lot of data and many classes.
# Regularizers ameliorate overfitting. They penalize overly complicated layers and large parameters.

input_shape = X.shape[1:]  # (128, frames, 1)

NUM_CLASSES = 112

inputs = layers.Input(shape=input_shape)

x = inputs

for filters in [64, 128, 256, 256]:
    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    
    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.2)(x)

# Hybrid pooling
x_avg = layers.GlobalAveragePooling2D()(x)
x_max = layers.GlobalMaxPooling2D()(x)
x = layers.Concatenate()([x_avg, x_max])

x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.4)(x)

outputs = layers.Dense(NUM_CLASSES, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs)

In [15]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.BinaryFocalCrossentropy(gamma=1.5),
    metrics=[
        tf.keras.metrics.AUC(curve='ROC', multi_label=True, name='roc_auc'),
        tf.keras.metrics.AUC(curve='PR', multi_label=True, name='pr_auc')
    ]
)

In [16]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_pr_auc',
        patience=8,
        min_delta=0.005,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    # This block reduces the learning rate near the solution
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        cooldown=1,
        mode='min',
        min_lr=1e-6,
        verbose=1
    ),
    
    tf.keras.callbacks.ModelCheckpoint(
        'best_model.keras',
        monitor='val_pr_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    )
]

In [19]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

2.21.0
[]


In [ ]:
# This program is too computationally intensive to run on a CPU.
# I ran it in Google Colab.

model.fit( # Send ChatGPT final val_pr_auc and y_pred.std() when this is done.
    X_train, y_train,
    batch_size=64,
    shuffle=True,
    validation_data=(X_val, y_val),
    epochs=30,
    callbacks=callbacks
)

Epoch 1/30
 17/668 ━━━━━━━━━━━━━━━━━━━━ 1:35:08 9s/step - loss: 0.6698 - pr_auc: 0.0147 - roc_auc: 0.4311

KeyboardInterrupt: 

In [56]:
# This model seems...questionable given how quickly it stopped and the data in the results below.
# Nevertheless, I will save it anyway and check it later.

model.save("best_common_bird_model.keras")

In [ ]:
y_pred = model.predict(X_val)
y_pred = y_pred ** 2
y_pred_scaled = np.clip((y_pred - 0.05) / 0.9, 0, 1)

print(y_pred_scaled.mean(axis=0))
print(y_pred_scaled.std(axis=0))

334/334 ━━━━━━━━━━━━━━━━━━━━ 34s 101ms/step
[0.08742873 0.13089475 0.13738088 0.14615981 0.11472994 0.09732453
 0.12758097 0.10633951 0.10180315 0.15720548 0.14506124 0.09330388
 0.13713752 0.09833556 0.08538619 0.09675778 0.09855196 0.11838873
 0.09066366 0.08855192 0.1304384  0.12066022 0.09064404 0.1511903
 0.14435877 0.11792654 0.12429129 0.11392086 0.1313905  0.13240048
 0.10983396 0.1287756  0.09859655 0.09688026 0.14371015 0.09813305
 0.14519069 0.11691119 0.13166633 0.10211068 0.11708012 0.10872786
 0.12555622 0.12536888 0.10287353 0.09586926 0.13812855 0.13226281
 0.09694994 0.11981555 0.12927149 0.10597144 0.14750782 0.10873806
 0.11827461 0.0910676  0.10460305 0.13441986 0.09828275 0.11388802
 0.12862657 0.10193391 0.09826835 0.10642099 0.08935876 0.09583482
 0.14589664 0.13455419 0.10057233 0.1167277  0.10725862 0.11863405
 0.11785601 0.1385289  0.10941649 0.10584556 0.09107208 0.11313358
 0.10195713 0.11175244 0.10699124 0.12049934 0.14269626 0.14129134
 0.13447152 0.09750

In [58]:
from sklearn.metrics import f1_score

thresholds = []

for i in range(NUM_CLASSES):
    best_t = 0.5
    best_f1 = 0

    for t in np.linspace(0.05, 0.9, 50):
        preds = (y_pred[:, i] > t).astype(int)
        f1 = f1_score(y_val[:, i], preds)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    thresholds.append(best_t)

print(thresholds)

[np.float64(0.2755102040816326), np.float64(0.29285714285714287), np.float64(0.25816326530612244), np.float64(0.2755102040816326), np.float64(0.25816326530612244), np.float64(0.22346938775510206), np.float64(0.2755102040816326), np.float64(0.24081632653061225), np.float64(0.18877551020408162), np.float64(0.24081632653061225), np.float64(0.22346938775510206), np.float64(0.17142857142857143), np.float64(0.24081632653061225), np.float64(0.2061224489795918), np.float64(0.22346938775510206), np.float64(0.2755102040816326), np.float64(0.24081632653061225), np.float64(0.2061224489795918), np.float64(0.15408163265306124), np.float64(0.25816326530612244), np.float64(0.17142857142857143), np.float64(0.18877551020408162), np.float64(0.18877551020408162), np.float64(0.2755102040816326), np.float64(0.29285714285714287), np.float64(0.2755102040816326), np.float64(0.22346938775510206), np.float64(0.15408163265306124), np.float64(0.22346938775510206), np.float64(0.31020408163265306), np.float64(0.2234

In [59]:
y_pred_binary = np.zeros_like(y_pred)

for i in range(NUM_CLASSES):
    y_pred_binary[:, i] = (y_pred[:, i] > thresholds[i]).astype(int)

In [60]:
from sklearn.metrics import classification_report

print(classification_report(y_val, y_pred_binary))

              precision    recall  f1-score   support

           0       1.00      0.48      0.65        46
           1       0.63      0.57      0.60       129
           2       0.39      0.42      0.40       175
           3       0.50      0.29      0.37       151
           4       0.81      0.55      0.65        84
           5       0.27      0.44      0.34        59
           6       0.52      0.35      0.42       152
           7       0.61      0.34      0.44        67
           8       0.18      0.17      0.18        58
           9       0.23      0.34      0.28       179
          10       0.21      0.26      0.23       166
          11       0.47      0.17      0.25        41
          12       0.39      0.41      0.40       145
          13       0.25      0.25      0.25        56
          14       0.32      0.27      0.30        44
          15       0.45      0.39      0.42        74
          16       0.28      0.30      0.29        57
          17       0.26    

/Users/noah/tf_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
